# Language Bridge · Final Project Checkpoint 2
**DESIGNTK 530 · Fall 2026**

**Language Bridge** helps people who speak a widely spoken language (or who know only a place, like a traveler or heritage learner) find endangered languages they can connect with. Revitalization programs reach new audiences, and users get a short, honest list of starting points.

This checkpoint shows working progress on:
- **Part 2 – Interface:** an interactive search UI (section 5) with a text input, filters, result cards, AI-estimate labels, loading feedback, and error and empty states.
- **Part 3 – LLM + agent:** a 5-step pipeline that **routes → retrieves → labels → checks → explains** (section 4).
- **Part 4 – Traditional ML:** **K-Means clustering + PCA** on the language embeddings (section 3). The clusters feed the agent, which uses them to diversify results and offer "explore nearby" suggestions.

Dataset: Endangered Languages Project data. Glottolog serves as the outside reference for checking AI labels.

Used Claude to help me with this assignment code.


## 1. Setup: key, client, data

In [ ]:
%pip install -q openai pandas numpy scikit-learn matplotlib ipywidgets

In [ ]:
import os, json, re
from getpass import getpass
from pathlib import Path
import urllib.request
import numpy as np, pandas as pd
from openai import OpenAI
from IPython.display import display, HTML

if "LITELLM_TOKEN" not in os.environ:
    os.environ["LITELLM_TOKEN"] = getpass("Enter your AI Gateway API key: ")
client = OpenAI(api_key=os.environ["LITELLM_TOKEN"], base_url="https://litellm.oit.duke.edu/v1")
CHAT_MODEL = "duke-current"
EMBED_MODEL, EMBED_DIMS = "text-embedding-3-small", 256

In [ ]:
data_path = Path("./endangered_languages.csv")
if not data_path.exists():
    urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1aJhMX-dsL1f-j8B9STFYuK8iFNEz34vx", data_path)

languages = pd.read_csv(data_path).rename(columns={
    "Name in English": "name", "Countries": "countries", "ISO639-3 codes": "iso_codes",
    "Degree of endangerment": "status", "Description of the location": "location"})
languages = languages[["ID", "name", "countries", "iso_codes", "status", "location"]]
languages = languages.dropna(subset=["name", "location"])
languages = languages[languages["status"] != "Extinct"]
languages["countries"] = languages["countries"].fillna("")
languages["iso_codes"] = languages["iso_codes"].fillna("unknown")
languages = languages.sample(500, random_state=42).reset_index(drop=True)   # same sample as I-9a, so saved vectors line up
languages["text"] = ("Language: " + languages["name"] + ". Countries: " + languages["countries"] +
                     ". Status: " + languages["status"] + ". Location: " + languages["location"])
print(languages.shape)

## 2. Embeddings (reuse I-9a vectors when available)

In [ ]:
def embed(texts):
    out = []
    for i in range(0, len(texts), 100):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i+100], dimensions=EMBED_DIMS)
        out.extend(d.embedding for d in sorted(r.data, key=lambda d: d.index))
    return np.asarray(out, dtype=float)

vec_path = Path("language_vectors.npy")
if vec_path.exists() and np.load(vec_path).shape[0] == len(languages):
    vectors = np.load(vec_path); print("Loaded saved vectors", vectors.shape)
else:
    vectors = embed(languages["text"].tolist()); np.save(vec_path, vectors); print("Embedded", vectors.shape)
unit_vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

## 3. Traditional ML: K-Means clustering + PCA (Part 4)

**Question it answers:** *Which endangered languages form natural groups (by region and description), and what else belongs with a given result?*

**Why it fits:** The data has no labels for "relatedness", so this is an unsupervised problem. K-Means on the embeddings groups languages whose records read alike, which mostly means geographic and areal neighbors. PCA reduces 256 dimensions to 2 for a map a user can read.

**Preprocessing:** combined text field → 256-d embedding → L2-normalized, so Euclidean K-Means behaves like cosine similarity. **Feature selection:** the embedding of name + countries + status + location. Status is also used later as a filter.

**Fitting:** I try k = 4…14 and pick k by **silhouette score**.

**How the output is used (consequential):**
1. The agent uses the cluster to **diversify** the top 5. It caps results at 2 per cluster, so one region cannot flood the list.
2. Each result card shows an **"Explore nearby"** row: other languages in the same cluster.
3. The PCA scatter is shown to the user as a map of the collection, with their results highlighted.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

scores = {}
for k in range(4, 15):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(unit_vectors)
    scores[k] = silhouette_score(unit_vectors, km.labels_)
best_k = max(scores, key=scores.get)
print("Silhouette by k:", {k: round(v, 3) for k, v in scores.items()})
print("Chosen k =", best_k)

kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(unit_vectors)
languages["cluster"] = kmeans.labels_
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(unit_vectors)
languages["pc1"], languages["pc2"] = xy[:, 0], xy[:, 1]
print("PCA variance explained (2 comps): %.1f%%" % (100 * pca.explained_variance_ratio_.sum()))

In [ ]:
# Interpret each cluster: its most common countries and status mix
def top_countries(s, n=3):
    c = s.str.split(",").explode().str.strip()
    return ", ".join(c[c != ""].value_counts().head(n).index)

cluster_summary = languages.groupby("cluster").agg(
    size=("name", "size"),
    top_countries=("countries", top_countries),
    pct_critical=("status", lambda s: round(100 * (s == "Critically endangered").mean(), 1)),
    examples=("name", lambda s: ", ".join(s.head(4))))
cluster_labels = cluster_summary["top_countries"].to_dict()   # short human-readable name per cluster
display(cluster_summary)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(languages.pc1, languages.pc2, c=languages.cluster, cmap="tab20", s=14, alpha=.8)
for c, g in languages.groupby("cluster"):
    ax.annotate(cluster_labels[c].split(",")[0], (g.pc1.mean(), g.pc2.mean()), fontsize=8, weight="bold")
ax.set_title(f"Endangered languages: K-Means (k={best_k}) on embeddings, PCA projection")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); plt.show()

**Limitations (with evidence):**
- The silhouette scores printed above are modest, so cluster boundaries are soft. Clusters are a *browsing aid*, not a classification.
- 2-D PCA keeps only the share of variance printed above, so points that look close on the map may not be close in the full space.
- Embeddings reflect *wording and geography*, not genealogy. In I-9a, the Russian search returned Uralic and Turkic languages, and the clusters show the same bias (they are mostly regional).
- The data is a 500-row sample, so some families and regions are thin.

## 4. The agent pipeline (Part 3)

```
user text ──► [1 ROUTE] LLM → JSON {intent, target, country}
                 │  intent = related_language | place | unclear | off_topic
                 ▼
             [2 RETRIEVE] embedding search (cosine) + country filter (place intent)
                 │  + status filter from UI + similarity threshold
                 ▼
             [3 DIVERSIFY] K-Means: max 2 results per cluster → top 5
                 ▼
             [4 LABEL + CHECK] LLM labels relative | family | confidence
                 │  checks: format parse, run twice → agree?, "unsure" → low,
                 │  relative == own name → flagged, family vs. target-language family
                 ▼
             [5 EXPLAIN] LLM writes a 1-2 sentence "how this connects to you" from the data only
                 ▼
             cards + warnings + Glottolog links  (or an honest "can't answer" state)
```

**Guardrails and failure modes**
- **Off-topic or unclear input:** the agent stops and asks a clarifying question instead of guessing.
- **Weak matches:** results below the similarity threshold are dropped. If none remain, the user sees an honest empty state.
- **Inconsistent labels:** each language is labeled twice. If the two runs disagree, the result is marked *"inconsistent – check Glottolog"* and confidence is downgraded.
- **Repeating the name** (e.g. Zapotec → "Zapotec"): flagged as uninformative.
- **API errors:** caught per step. The UI shows an error message and keeps the dataset fields visible.
- **Every AI field is labeled "AI estimate"** and links to Glottolog by ISO code so users can verify it.

In [ ]:
def ask_llm(prompt):
    r = client.responses.create(model=CHAT_MODEL, input=prompt)
    return r.output_text.strip()

# ---------- Step 1: ROUTE ----------
ROUTER_PROMPT = '''You route searches for an endangered-language discovery app.
Classify the user's request and reply with JSON only, no prose:
{"intent": "related_language" | "place" | "unclear" | "off_topic",
 "target": "<widely spoken language the user knows, or null>",
 "country": "<country/region named, in English, or null>",
 "clarify": "<one short question if intent is unclear, else null>"}
- related_language: user wants languages related to a language they know (e.g. "I speak Spanish").
- place: user names a place (travel, heritage, "my grandparents were from Sicily").
User request: '''

def route(user_text):
    raw = ask_llm(ROUTER_PROMPT + user_text)
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        plan = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        plan = {}
    if plan.get("intent") not in {"related_language", "place", "unclear", "off_topic"}:
        plan = {"intent": "unclear", "target": None, "country": None,
                "clarify": "Could you name a language you speak or a place you're interested in?"}
    return plan

# ---------- Step 2 + 3: RETRIEVE + DIVERSIFY ----------
MIN_SIM = 0.25   # below this, a match is too weak to show

def retrieve(user_text, plan, status_filter="All", k=5):
    q = embed([user_text])[0]; q = q / np.linalg.norm(q)
    sims = unit_vectors @ q
    cand = languages.assign(similarity=sims)
    if status_filter != "All":
        cand = cand[cand.status == status_filter]
    if plan.get("intent") == "place" and plan.get("country"):
        in_country = cand[cand.countries.str.contains(plan["country"], case=False, na=False)]
        if len(in_country):          # hard filter only if the country exists in the data
            cand = in_country
    cand = cand[cand.similarity >= MIN_SIM].sort_values("similarity", ascending=False)
    picked, per_cluster = [], {}
    for idx, r in cand.iterrows():   # K-Means diversity cap
        if per_cluster.get(r.cluster, 0) < 2:
            picked.append(idx); per_cluster[r.cluster] = per_cluster.get(r.cluster, 0) + 1
        if len(picked) == k: break
    return cand.loc[picked]

# ---------- Step 4: LABEL + CHECK ----------
LABEL_PROMPT = ("You are labeling endangered languages. Using the language name, countries, ISO code, and location below, "
    "name the most closely related widely spoken language and the language family. Use your own knowledge of linguistics. "
    "If you are not sure, write 'unsure' and set confidence to low. Reply in exactly this format and nothing else: "
    "relative | family | confidence. Confidence must be high, medium, or low.")

def parse_label(txt):
    p = [x.strip() for x in txt.split("|")]
    return p if len(p) == 3 else None

def label_and_check(row, target=None):
    prompt = LABEL_PROMPT + f"\n\nLanguage: {row['name']}\nCountries: {row['countries']}\nISO code: {row['iso_codes']}\nLocation: {row['location']}"
    runs = []
    for _ in range(2):                      # self-consistency check
        try: runs.append(parse_label(ask_llm(prompt)))
        except Exception: runs.append(None)
    flags = []
    good = [r for r in runs if r]
    if not good:
        return {"relative": "unavailable", "family": "unavailable", "confidence": "low", "flags": ["AI label failed"]}
    rel, fam, conf = good[0]
    conf = conf.lower() if conf.lower() in {"high", "medium", "low"} else "low"
    if len(good) == 2 and (good[0][0].lower() != good[1][0].lower() or good[0][1].lower() != good[1][1].lower()):
        flags.append(f"inconsistent between runs ({good[0][0]} vs {good[1][0]}) – check Glottolog")
        conf = "low"
    if rel.lower() == "unsure":
        conf = "low"; flags.append("model unsure")
    if rel.lower() in row["name"].lower():
        flags.append("relative just repeats the language name")
        if conf == "high": conf = "medium"
    if target and rel.lower() != "unsure" and target.lower() not in (rel + fam).lower():
        flags.append(f"not clearly related to {target}; shown as a nearby/regional match")
    return {"relative": rel, "family": fam, "confidence": conf, "flags": flags}

# ---------- Step 5: EXPLAIN ----------
def explain(row, lab, plan, user_text):
    prompt = (f"In 1-2 friendly sentences, tell this user how the endangered language below might connect to them. "
              f"Use ONLY the facts given; say it is an estimate if confidence is low. No made-up facts.\n"
              f"User said: {user_text}\nLanguage: {row['name']} ({row['status']}), spoken in {row['countries']} – {row['location']}.\n"
              f"AI estimate: related to {lab['relative']}, family {lab['family']}, confidence {lab['confidence']}.")
    try: return ask_llm(prompt)
    except Exception: return "Explanation unavailable right now. The dataset fields above are still accurate."

# ---------- Orchestrator ----------
def run_agent(user_text, status_filter="All", log=print):
    trace = []
    if not user_text.strip():
        return {"state": "empty_input", "trace": trace}
    log("1/5 Routing your request…"); plan = route(user_text); trace.append(("route", plan))
    if plan["intent"] in {"unclear", "off_topic"}:
        return {"state": "clarify", "plan": plan, "trace": trace}
    log("2/5 Searching 500 language records…"); hits = retrieve(user_text, plan, status_filter)
    trace.append(("retrieve", len(hits)))
    if hits.empty:
        return {"state": "no_results", "plan": plan, "trace": trace}
    results = []
    for i, (_, row) in enumerate(hits.iterrows(), 1):
        log(f"3/5 Labeling + checking {i}/{len(hits)}: {row['name']}…")
        lab = label_and_check(row, plan.get("target"))
        log(f"4/5 Explaining {row['name']}…")
        expl = explain(row, lab, plan, user_text)
        nearby = languages[(languages.cluster == row.cluster) & (languages.ID != row.ID)].name.head(3).tolist()
        results.append({**row.to_dict(), **lab, "explanation": expl, "nearby": nearby})
    log("5/5 Done.")
    return {"state": "ok", "plan": plan, "results": results, "trace": trace}

### Quick test of the router (no UI)

In [ ]:
for q in ["I speak Spanish", "I'm travelling to Italy next month", "my grandparents were from Chukotka", "what's the weather"]:
    print(q, "→", route(q))

## 5. Interface prototype (Part 2)

The interface has a text box, a status filter, and a Search button. A live progress line shows which agent step is running. Each result card shows **dataset facts** (name, place, status) separately from the **AI estimate** (relative, family, confidence), along with warning flags, a Glottolog link, and an "Explore nearby" row from the K-Means clusters. Empty, clarify, and error states each have their own message.

In [ ]:
import html as _h
STATUS_COLOR = {"Vulnerable": "#2e7d32", "Definitely endangered": "#f9a825",
                "Severely endangered": "#ef6c00", "Critically endangered": "#c62828"}
CONF_COLOR = {"high": "#2e7d32", "medium": "#f9a825", "low": "#c62828"}

def card(r):
    e = lambda s: _h.escape(str(s))
    iso = str(r["iso_codes"]).split(",")[0].strip()
    link = f"https://glottolog.org/glottolog?iso={iso}" if iso != "unknown" else f"https://glottolog.org/glottolog?name={e(r['name'])}"
    flags = "".join(f"<li>⚠️ {e(f)}</li>" for f in r["flags"])
    return f'''<div style="border:1px solid #ddd;border-radius:10px;padding:14px;margin:10px 0;font-family:sans-serif">
      <div style="display:flex;justify-content:space-between;align-items:center">
        <b style="font-size:17px">{e(r['name'])}</b>
        <span style="background:{STATUS_COLOR.get(r['status'],'#777')};color:white;padding:2px 8px;border-radius:8px;font-size:12px">{e(r['status'])}</span></div>
      <div style="color:#555;font-size:13px;margin:4px 0">📍 {e(r['countries'])} – {e(r['location'])[:160]}</div>
      <div style="background:#f5f3ff;border-left:4px solid #7c4dff;padding:8px;margin:8px 0;font-size:13px">
        <span style="font-size:11px;color:#7c4dff;font-weight:bold">AI ESTIMATE · verify before relying on it</span><br>
        Related to <b>{e(r['relative'])}</b> · Family <b>{e(r['family'])}</b> ·
        Confidence <b style="color:{CONF_COLOR[r['confidence']]}">{e(r['confidence'])}</b>
        <ul style="margin:4px 0 0 0;padding-left:18px;color:#8d6e00">{flags}</ul></div>
      <div style="font-size:14px"><b>How it connects to you:</b> {e(r['explanation'])}</div>
      <div style="font-size:12px;color:#555;margin-top:8px">🔎 Explore nearby (same cluster): {e(', '.join(r['nearby']) or '—')} ·
        <a href="{link}" target="_blank">Check on Glottolog ↗</a> · match score {r['similarity']:.2f}</div></div>'''

def render(res):
    s = res["state"]
    if s == "empty_input":
        return "<p>✏️ Type a language you speak or a place you care about to get started.</p>"
    if s == "clarify":
        return f"<p>🤔 {_h.escape(res['plan'].get('clarify') or 'Could you name a language you speak or a place?')}</p>"
    if s == "no_results":
        return ("<p>😕 <b>No reliable matches.</b> This sample of 500 languages has nothing close enough to your search. "
                "Try a broader place or another language, or remove the status filter.</p>")
    p = res["plan"]
    head = (f"<p style='font-family:sans-serif'>Interpreted as: <b>{p['intent'].replace('_',' ')}</b>"
            + (f" · language: <b>{_h.escape(str(p['target']))}</b>" if p.get('target') else "")
            + (f" · place: <b>{_h.escape(str(p['country']))}</b>" if p.get('country') else "") + "</p>")
    return head + "".join(card(r) for r in res["results"])

def highlight_plot(res):
    if res.get("state") != "ok": return
    ids = [r["ID"] for r in res["results"]]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(languages.pc1, languages.pc2, c="#ddd", s=10)
    sel = languages[languages.ID.isin(ids)]
    ax.scatter(sel.pc1, sel.pc2, c="#7c4dff", s=60, edgecolor="k")
    for _, r in sel.iterrows(): ax.annotate(r["name"], (r.pc1, r.pc2), fontsize=8)
    ax.set_title("Your results on the language map (PCA of embeddings)"); ax.set_xticks([]); ax.set_yticks([]); plt.show()

In [ ]:
try:
    import ipywidgets as w
    query = w.Text(placeholder='e.g. "I speak Spanish" or "travelling to Italy"', layout=w.Layout(width="420px"))
    status = w.Dropdown(options=["All"] + sorted(languages.status.unique()), description="Status:")
    go = w.Button(description="Find languages", button_style="primary", icon="search")
    progress = w.HTML(); out = w.Output()

    def on_click(_):
        out.clear_output(); go.disabled = True
        try:
            res = run_agent(query.value, status.value, log=lambda m: setattr(progress, "value", f"<i>{m}</i>"))
            with out:
                display(HTML(render(res))); highlight_plot(res)
        except Exception as ex:
            with out:
                display(HTML(f"<p style='color:#c62828'>⚠️ Something went wrong reaching the AI service ({_h.escape(type(ex).__name__)}). "
                             "Check your API key or connection and try again.</p>"))
        finally:
            go.disabled = False; progress.value = ""
    go.on_click(on_click); query.on_submit(on_click)
    display(w.HTML("<h2 style='font-family:sans-serif'>🌍 Language Bridge</h2>"
                   "<p style='font-family:sans-serif'>Find endangered languages connected to a language you know or a place you care about. "
                   "Purple boxes are <b>AI estimates</b>; everything else comes from the Endangered Languages Project dataset.</p>"),
            w.HBox([query, go]), status, progress, out)
except ImportError:
    print("ipywidgets not installed – run: %pip install ipywidgets   (fallback below)")
    def search(text, status_filter="All"):
        res = run_agent(text, status_filter); display(HTML(render(res))); highlight_plot(res)
    search("I speak Spanish")

### Demo runs (one per user type, for the record)

In [ ]:
for q in ["I speak Italian and want to learn something related", "I'm travelling to Mexico", "asdf"]:
    print("=" * 80, "\nQUERY:", q)
    res = run_agent(q)
    display(HTML(render(res)))

## 6. Consequential UX / product decisions (and how they changed)

1. **Separate dataset facts from AI estimates visually.** In I-9a the model answered "Dogri" in one run and "Hindi" in another for Baghati, and labeled "Zapotec" for Zapotec as *high* confidence. *Revision:* AI fields now sit in a purple "AI ESTIMATE" box. Each language is labeled twice, disagreements are flagged, and confidence is downgraded automatically. Each card also links to Glottolog.
2. **Diversify results with K-Means.** Plain cosine search returned clusters of very similar records (e.g. three Vanuatu languages all labeled "Fijian"). *Revision:* results are capped at 2 per cluster, and the rest of the cluster moves to "Explore nearby". The top 5 is more varied, and the user can still drill down.
3. **Route before searching.** Early tests treated "Italy" (a place) and "Italian" (a language) the same way. The router now extracts intent and country first. Place queries get a hard country filter, and vague input gets a clarifying question instead of 5 guesses.
4. **Honest empty states.** A similarity threshold means the app says "no reliable matches" instead of padding the list with weak results.

## 7. Next steps
- Build a Glottolog benchmark (≈50 hand-checked rows) to measure label accuracy and tune `MIN_SIM`.
- Add links to revitalization programs and learning resources as the call to action.
- Move the UI to a Streamlit/web prototype, and run user tests with 3–5 people from the target groups.